# Step 12 · Resolve GBIF references

Resolve and checkpoint GBIF links for pathogen species included in the accepted native Explorer profiles.


## Step 1 · Load the resolved run and taxonomy


In [1]:
import json, sys
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    raise RuntimeError('Open this notebook from the Pathogen Atlas V4 repository.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'notebooks'))
from _shared import configure_notebook, configured_run_id
RUN_ID = configured_run_id(PROJECT_ROOT)
ROOT, LOG, LOG_PATH = configure_notebook('12_resolve_gbif_references', RUN_ID)
RUN_PATH = ROOT / 'data/v4/runs' / RUN_ID / 'resolved_config.json'
if not RUN_PATH.exists():
    raise FileNotFoundError(f'Run notebook 00 first; resolved selection is missing: {RUN_PATH}')
resolved_run = json.loads(RUN_PATH.read_text(encoding='utf-8'))
ACCEPTED_PATH = ROOT / 'data/v4/runs' / RUN_ID / 'release/pathogenic_interest_index.parquet'
if not ACCEPTED_PATH.exists():
    raise FileNotFoundError(f'Run notebook 05 first; accepted Explorer profiles are missing: {ACCEPTED_PATH}')
pathogen_ids = set(pd.read_parquet(ACCEPTED_PATH, columns=['pathogen_id']).pathogen_id.astype(str))
TAXONOMY_ROOT = sorted((ROOT / 'data/v4/staging').glob('*/taxonomy'))[-1]
RUN_TAXONOMY_ROOT = ROOT / 'data/v4/runs' / RUN_ID / 'taxonomy'
PATHOGEN_TABLE = RUN_TAXONOMY_ROOT / 'pathogens.parquet'
if not PATHOGEN_TABLE.exists():
    raise FileNotFoundError('Run taxonomy was not resolved; run 00_configure_run.ipynb first.')
pathogens = pd.read_parquet(PATHOGEN_TABLE, columns=['pathogen_id','pathogen_group','accepted_name','rank'])
scope = pathogens.loc[pathogens.pathogen_id.astype(str).isin(pathogen_ids) & pathogens['rank'].astype(str).str.casefold().eq('species')]
PATHOGEN_GROUPS = sorted(scope.pathogen_group.astype(str).unique())
OUTPUT_PATH = ROOT / 'data/v4/references/release/12_resolve_gbif_references.csv'
FORCE_REFRESH = False
existing = pd.read_csv(OUTPUT_PATH, keep_default_na=False) if OUTPUT_PATH.exists() else pd.DataFrame()
display(Markdown(f"**Run:** `{RUN_ID}` · **Groups:** {', '.join(PATHOGEN_GROUPS)} · **Accepted species profiles:** {scope.pathogen_id.nunique():,} · **Existing rows:** {len(existing):,} · **Force refresh:** {FORCE_REFRESH}"))
LOG.info('Reference lookup prepared: run=%s groups=%s species=%d existing=%d', RUN_ID, PATHOGEN_GROUPS, scope.pathogen_id.nunique(), len(existing))


2026-11-02 21:50 INFO Notebook session started at 2026-11-02T21:50:15.610762+00:00


2026-11-02 21:50 INFO Project root: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4


2026-11-02 21:50 INFO Run log: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/logs/pilot-v1/12_resolve_gbif_references.log


**Run:** `pilot-v1` · **Groups:** viruses · **Accepted species profiles:** 3 · **Existing rows:** 0 · **Force refresh:** False

2026-11-02 21:50 INFO Reference lookup prepared: run=pilot-v1 groups=['viruses'] species=3 existing=0


## Step 2 · Resolve and save GBIF associations

The lookup writes one row per pathogen ID, saves progress after every request batch, and reuses completed rows on rerun.

In [2]:
from pathogen_atlas_v4.references.links import resolve_gbif_links
service = 'GBIF'

links = resolve_gbif_links(PATHOGEN_TABLE, OUTPUT_PATH, PATHOGEN_GROUPS, force_refresh=FORCE_REFRESH, pathogen_ids=pathogen_ids)
found = links.match_status.eq('found')
display(Markdown(f"**Saved:** `{OUTPUT_PATH.relative_to(ROOT)}` · **Records:** {len(links):,} · **{service} references found:** {int(found.sum()):,} · **No exact reference:** {int((~found).sum()):,}"))
display(links.head(12))
LOG.info('Reference lookup complete: service=GBIF records=%d found=%d output=%s', len(links), int(found.sum()), OUTPUT_PATH)


Resolving 3 pathogen records; reusing 0 completed rows.


Resolved 3/3 pending records.


Saved 3 associations to /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/references/release/12_resolve_gbif_references.csv


**Saved:** `data/v4/references/release/12_resolve_gbif_references.csv` · **Records:** 3 · **GBIF references found:** 1 · **No exact reference:** 2

,pathogen_id,pathogen_group,accepted_name,gbif_taxon_key,gbif_scientific_name,gbif_url,match_status,match_type,confidence
0,viruses:976f27c4e726cd4225b8,viruses,Polerovirus RDPV,,,,not_found,NONE,100
1,viruses:f77114fb0e9d224e3584,viruses,Tungrovirus oryzae,12200616,Tungrovirus oryzae,https://www.gbif.org/species/12200616,found,EXACT,100
2,viruses:fa9873798bea5617f432,viruses,Sobemovirus RYMV,,,,not_found,NONE,100


2026-11-02 21:50 INFO Reference lookup complete: service=GBIF records=3 found=1 output=/Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/references/release/12_resolve_gbif_references.csv
